<a href="https://colab.research.google.com/github/jakovitz-W/PoDD-Optimization/blob/minor-save_points/CS_511_Semester_Project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import drive

drive.mount("/content/data")

Mounted at /content/data


In [2]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
import torch.nn.functional as F

import time
from time import perf_counter as pc
import pandas as pd

In [20]:
#load datasets
transform = transforms.Compose(
    [transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))])

def GetData(dataset):
  if(dataset == 'CIFAR10'):
    trainset = torchvision.datasets.CIFAR10(root='./data/MyDrive/ECE 511', train=True, download=False, transform=transform)
    trainloader = torch.utils.data.DataLoader(trainset, batch_size=4, shuffle=True, num_workers=2)
    testSet = torchvision.datasets.CIFAR10(root='./data/MyDrive/ECE 511', train=False, download=False, transform=transform)
    testLoader = torch.utils.data.DataLoader(testSet, batch_size=4, shuffle=False, num_workers=2)
    classes = ('plane', 'car', 'bird', 'cat',
           'deer', 'dog', 'frog', 'horse', 'ship', 'truck')

  if(dataset == 'CIFAR100'):
    trainset = torchvision.datasets.CIFAR100(root='./data/MyDrive/ECE 511', train=True, download=False, transform=transform)
    trainloader = torch.utils.data.DataLoader(trainset, batch_size=4, shuffle=True, num_workers=2)
    testSet = torchvision.datasets.CIFAR100(root='./data/MyDrive/ECE 511', train=False, download=False, transform=transform)
    testLoader = torch.utils.data.DataLoader(testSet, batch_size=4, shuffle=False, num_workers=2)

#GetData('CIFAR10')
#GetData('CIFAR100')

trainset = torchvision.datasets.CIFAR10(root='./data/MyDrive/ECE 511', train=True, download=False, transform=transform)
trainloader = torch.utils.data.DataLoader(trainset, batch_size=4, shuffle=True, num_workers=2)
testSet = torchvision.datasets.CIFAR10(root='./data/MyDrive/ECE 511', train=False, download=False, transform=transform)
testLoader = torch.utils.data.DataLoader(testSet, batch_size=4, shuffle=False, num_workers=2)
classes = ('plane', 'car', 'bird', 'cat',
      'deer', 'dog', 'frog', 'horse', 'ship', 'truck')

In [8]:
class Net(nn.Module):
  def __init__(self):
    super(Net, self).__init__()
    self.conv1 = nn.Conv2d(3,6,5)
    self.pool = nn.MaxPool2d(2,2)
    self.conv2 = nn.Conv2d(6, 16, 5)
    self.fc1 = nn.Linear(16 * 5 * 5, 120)
    self.fc2 = nn.Linear(120, 84)
    self.fc3 = nn.Linear(84, 10)

  def forward(self, x):
    x = self.pool(F.relu(self.conv1(x)))
    x = self.pool(F.relu(self.conv2(x)))
    x = x.view(-1, 16 * 5 * 5)
    x = F.relu(self.fc1(x))
    x = F.relu(self.fc2(x))
    x = self.fc3(x)
    return x

net = Net()


In [47]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.SGD(net.parameters(), lr=0.001, momentum=0.9)
metrics_df = pd.DataFrame(columns = ["Version", "Epoch", "Time"])

In [46]:
def Train(epochs):

  for epoch in range(epochs):
    t1 = pc()
    running_loss = 0.0
    for i, data in enumerate(trainloader, 0):
      inputs, labels = data

      optimizer.zero_grad()

      outputs = net(inputs)
      loss = criterion(outputs, labels)
      loss.backward()
      optimizer.step()

      running_loss += loss.item()
      if i % 2000 == 1999:
          print('[%d, %5d] loss: %.3f' %
                (epoch + 1, i + 1, running_loss / 2000))
          running_loss = 0.0
    t2 = pc()

    metrics_df = LoadMetrics("CIFAR10_Plain", epoch, t2-t1)
  metrics_df

In [ ]:

#stores for local session
def LoadMetrics(version, epoch, time):
  df2 = pd.DataFrame([[version, epoch, time]], columns=["Version", "Epoch", "Time"])
  return pd.concat([metrics_df, df2], ignore_index=True)
#todo: append data to csv file & store

In [ ]:
Train(20)

In [45]:
correct = 0
total = 0

with torch.no_grad():
  for data in testLoader:
      images, labels = data
      outputs = net(images)
      _, predicted = torch.max(outputs.data, 1)
      total += labels.size(0)
      correct += (predicted == labels).sum().item()
print('Accuracy: %d %%' %(100 * correct / total))

Accuracy: 63 %
